In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pytz import timezone

In [0]:
def get_arguments():
    try:
        args = {
            "databricks_env": os.environ["Databricks_Env"],
            "loadtracker_target_catalog": os.environ["Loadtracker_Target_Catalog"],
            "loadtracker_target_schema": os.environ["Loadtracker_Target_Schema"],
            "loadtracker_target_table": os.environ["Loadtracker_Target_Table"],
            "datamapping_file": os.environ["DataMapping_File"]
        }

        print("Arguments retrieved:", args)  # Added print statement
        return args
    except Exception as e:
        raise Exception("Error: Failed to get_arguments - " + str(e))

In [0]:
def get_current_time():
    try:
        tz = timezone(f"EST")
        DATE = datetime.now(tz)
        
        return DATE
    except Exception as e:
        raise Exception("Error: Failed to get_current_time - "+str(e))

In [0]:
def get_entity_config_mapping(args):
    try:
        file_path = f"./{args['datamapping_file']}"

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Data mapping file not found: {file_path}")

        with open(file_path, "r") as f:
            json_data = json.load(f)
        
        # Transform JSON data into a list of dictionaries
        entities = [{"entity": key, "mapping": value} for key, value in json_data.items()]
        
        return entities
    except FileNotFoundError as fnf_error:
        raise Exception(f"Error: File not found - {str(fnf_error)}")
    except Exception as e:
        raise Exception("Error: Failed to get_entity_config_mapping - " + str(e))

In [0]:
def extract_to_adls_new(entity, args):
    try:
        print("Executing Runner Notebook for " + entity['entity'])
        dict_sql = json.dumps(entity['mapping'])
        
        #Run the Runner notebook and capture the response
        runstatus = dbutils.notebook.run(
            f"./Runner",
            30000,
            {
                "DatabricksEnv": args["databricks_env"],
                "monitoringcatalog": args["loadtracker_target_catalog"],
                "monitoringschema": args["loadtracker_target_schema"],
                "monitoringtable": args["loadtracker_target_table"],
                "dict_sql": dict_sql
            }
        )

        # Parse the runstatus to handle JSON properly
        response = json.loads(runstatus)
        if response.get("status") == "SUCCESS":
            print("Notebook executed successfully.")
        elif response.get("status") == "ERROR":
            print(f"Notebook failed with error: {response.get('error')}")
            error_message = f"Runner failed for {entity.entity} with status: {response}"
            raise Exception(error_message)
        else:
            raise Exception(f"Unexpected response from Runner: {response}")
    except Exception as e:
        # Raise a clean exception with context
        raise Exception(f"Error: Failed to extract_to_adls_new for {entity.entity} - {str(e)}")

In [0]:
def concurrent_extracts_to_adls_new(entities, args):
    try:
        with ThreadPoolExecutor() as executor:
            futures = [executor.submit(extract_to_adls_new, Key, args) for Key in entities]
    except Exception as e:
        raise Exception("Error: Failed to concurrent_extracts_to_adls_new - " + str(e))

In [0]:
def create_monitoring_table(args, desired_schema):
    """
    Ensures that the table exists with the desired schema.
    Only creates the table if it does not already exist.
    """
    try:
        # Define the full table name
        full_table_name = f"{args['loadtracker_target_catalog']}.{args['loadtracker_target_schema']}.{args['loadtracker_target_table']}"

        # Check if the table exists
        table_exists_query = f"""
            SHOW TABLES IN {args['loadtracker_target_catalog']}.{args['loadtracker_target_schema']}
            LIKE '{args['loadtracker_target_table']}'
        """
        table_exists = spark.sql(table_exists_query).count() > 0

        if table_exists:
            print(f"Table {full_table_name} already exists.")
            return

        # Create the table with the desired schema
        schema_definition = ", ".join([f"{col} {dtype}" for col, dtype in desired_schema.items()])
        spark.sql(f"""
            CREATE TABLE {full_table_name} (
                {schema_definition}
            ) USING DELTA
        """)
        print(f"Table {full_table_name} created successfully.")
    except Exception as e:
        raise Exception(f"Error: Failed to ensure table schema - {str(e)}")

In [0]:
desired_schema = {
    "TABLE_NAME": "STRING",
    "LOAD_DATE": "TIMESTAMP",  # Changed from DATE to TIMESTAMP
    "LOAD_COMPLETE_FLAG": "STRING",
    "ROW_COUNT": "INT",
    "LAST_LOAD_TIMESTAMP": "TIMESTAMP"
}

In [0]:
try:
    args = get_arguments()
    create_monitoring_table(args, desired_schema)
    entities = get_entity_config_mapping(args)
    concurrent_extracts_to_adls_new(entities, args)
except Exception as e:
    print(f"Job failed with error: {str(e)}")
    dbutils.notebook.exit(json.dumps({"status": "ERROR", "error": str(e)}))
    raise

Arguments retrieved: {'databricks_env': 'dev', 'loadtracker_target_catalog': 'ent_dtlk_dev', 'loadtracker_target_schema': 'monitoring', 'loadtracker_target_table': 'load_tracking_s0transportxxx', 'datamapping_file': 'datamapping_Daily.json'}
Table ent_dtlk_dev.monitoring.load_tracking_s0transportxxx already exists.
Executing Runner Notebook for DeliverySchedule
Executing Runner Notebook for CubePalletConversion
Executing Runner Notebook for TempRSMSchedule
